In [1]:
import numpy as np
import warnings
import matplotlib.pyplot as plt
import xarray as xr
import cmocean
from dask.distributed import Client

from xmitgcm import llcreader
import ecco_v4_py

# Dataset

In [2]:
aste = llcreader.CRIOSPortalASTE270Model()

ds = aste.get_dataset()
ds = ds.sel(time='2002-03')

ds = ds.rename({'face':'tile'})
cds = ds.coords.to_dataset().reset_coords()
ds = ds.reset_coords(drop=True)

xgcmgrd = ecco_v4_py.get_llc_grid(cds,domain='aste')

# Constants

In [3]:
rhoconst = 1029
rA = cds.rA

# Horizontal Diffusion

In [4]:
DFxE_SLT = ds.DFxE_SLT.chunk((1, 10, 1, 90, 90))
DFyE_SLT = ds.DFyE_SLT.chunk((1, 10, 1, 90, 90))

DFxE_SLT = DFxE_SLT*rhoconst
DFyE_SLT = DFyE_SLT*rhoconst

DFxyE_SLT_diff = xgcmgrd.diff_2d_vector({'X' : DFxE_SLT.fillna(0), 
                                         'Y' : DFyE_SLT.fillna(0)},
                                        boundary = 'fill')
DFxE_SLT_diffx = DFxyE_SLT_diff['X']
DFyE_SLT_diffy = DFxyE_SLT_diff['Y']

dif_hConvS = (-(DFxE_SLT_diffx + DFyE_SLT_diffy)).transpose('time','tile','k','j','i')

# Horizontal Advection

In [5]:
ADVx_SLT = ds.ADVx_SLT.chunk((1, 10, 1, 90, 90))
ADVy_SLT = ds.ADVy_SLT.chunk((1, 10, 1, 90, 90))

ADVx_SLT = ADVx_SLT*rhoconst
ADVy_SLT = ADVy_SLT*rhoconst

ADVxy_SLT_diff = xgcmgrd.diff_2d_vector({'X' : ADVx_SLT.fillna(0),
                                         'Y' : ADVy_SLT.fillna(0)},
                                        boundary = 'fill')
ADVx_SLT_diffx = ADVxy_SLT_diff['X']
ADVy_SLT_diffy = ADVxy_SLT_diff['Y']

adv_hConvS = (-(ADVx_SLT_diffx + ADVy_SLT_diffy)).transpose('time','tile','k','j','i')

# Horizontal Convergence

In [6]:
hconv = adv_hConvS+dif_hConvS
print(hconv.sum().compute().values)

-0.000732421875


# Forcing

In [7]:
SFLUX    = ds.SFLUX.assign_coords(k=-5).expand_dims('k').chunk((1, 10, 1, 90, 90)).transpose('time','tile','k','j','i')
oceSPflx = ds.oceSPflx.assign_coords(k=-5).expand_dims('k').chunk((1, 10, 1, 90, 90)).transpose('time','tile','k','j','i')
oceSPtnd = ds.oceSPtnd.chunk((1, 10, 1, 90, 90)).transpose('time','tile','k','j','i')

budgO_zconv = (SFLUX+oceSPflx)*rA

tmp1 = (oceSPflx.values-oceSPtnd.cumsum(dim='k'))*rA

Forc_SLT = xr.concat([budgO_zconv,tmp1[:,:,:-1]],dim='k')
Forc_SLT = Forc_SLT.rename({'k':'k_l'})
Forc_SLT = Forc_SLT.assign_coords(k_l=cds['k_l'])
Forc_SLT = Forc_SLT.fillna(0)

forcS = -xgcmgrd.diff(Forc_SLT, 'Z', boundary='fill')

# Vertical Diffusion

In [8]:
DFrE_SLT = ds.DFrE_SLT.chunk((1, 10, 1, 90, 90))
DFrI_SLT = ds.DFrI_SLT.chunk((1, 10, 1, 90, 90))
KPPg_SLT = ds.KPPg_SLT.chunk((1, 10, 1, 90, 90))

diffV_vars = (DFrE_SLT+DFrI_SLT+KPPg_SLT)*rhoconst

dif_vConvS = xgcmgrd.diff(diffV_vars.fillna(0), 'Z', boundary='fill').transpose('time','tile','k','j','i')

# Vertical Advection

In [9]:
ADVr_SLT = ds.ADVr_SLT.chunk((1, 10, 1, 90, 90))

advV_vars = ADVr_SLT*rhoconst

adv_vConvS = xgcmgrd.diff(advV_vars.fillna(0), 'Z', boundary='fill').transpose('time','tile','k','j','i')

# Vertical Convergence

In [10]:
zconv = forcS+adv_vConvS+dif_vConvS
print(zconv.sum().compute().values)

-3157417150.250061


# Tendency

In [11]:
dt = 2419200.0
saltdr = xr.open_dataset('/home/jbonil11/vast_thaine1/jbonil11/ASTE/DATA/SALTDR7.zarr')
saltdr = saltdr.rename({'face':'tile'})

#changing the area array into a 4-face structure
#to match my snapshot data
rA2 = rA.copy()
rA2 = rA2.isel(tile=np.arange(1,5))
rA2 = rA2.assign_coords(tile=np.arange(saltdr.sizes["tile"]))

SALTsnap = saltdr['SALTDR'][0:2].chunk((1, 10, 1, 90, 90))
SALTsnap = SALTsnap.where(SALTsnap.mean(dim='time_snap') != 0, np.nan)

tendS = SALTsnap.diff('time_snap')/dt
tendS = tendS*rhoconst*rA2
tendS = tendS.rename({'time_snap':'time'})
tendS = tendS.assign_coords(time=forcS['time'])
tendS = tendS.transpose('time','tile','k','j','i')

/home/jbonil11/miniconda3/envs/astebath/lib/python3.10/site-packages/xarray/backends/plugins.py:139: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/home/jbonil11/miniconda3/envs/astebath/lib/python3.10/site-packages/xarray/backends/plugins.py:139: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


In [12]:
print(tendS[0].sum().compute().values)

-626647985.8525715


In [ ]:
#changing the terms into a 4 face structure to match the
#tendency
forcS4 = forcS.isel(tile=np.arange(1,5))
forcS4 = forcS4.assign_coords(tile=np.arange(saltdr.sizes["tile"]))

adv_vConvS4 = adv_vConvS.isel(tile=np.arange(1,5))
adv_vConvS4 = adv_vConvS4.assign_coords(tile=np.arange(saltdr.sizes["tile"]))

adv_hConvS4 = adv_hConvS.isel(tile=np.arange(1,5))
adv_hConvS4 = adv_hConvS4.assign_coords(tile=np.arange(saltdr.sizes["tile"]))

dif_vConvS4 = dif_vConvS.isel(tile=np.arange(1,5))
dif_vConvS4 = dif_vConvS4.assign_coords(tile=np.arange(saltdr.sizes["tile"]))

dif_hConvS4 = dif_hConvS.isel(tile=np.arange(1,5))
dif_hConvS4 = dif_hConvS4.assign_coords(tile=np.arange(saltdr.sizes["tile"]))

res = tendS - forcS4 - adv_vConvS4 - adv_hConvS4 - dif_vConvS4 - dif_hConvS4

In [ ]:
from matplotlib.colors import BoundaryNorm
a = res[0]
fig, axs = plt.subplots(2, 2, figsize=(11, 9))

vmax = 2.786517143249512e-07
vmin = -2.786517143249512e-07
cmap = plt.get_cmap('seismic')
levels = np.linspace(vmin, vmax, 22)  # num_levels bins
#levels = np.delete(levels,10)
norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, clip=True)

g = axs[0, 0].pcolormesh(a[0,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-2e-7, -1e-7, 0, 1e-7,2e-7])
g = axs[0, 1].pcolormesh(a[1,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-2e-7, -1e-7, 0, 1e-7,2e-7])
axs[0, 1].set_title('Salt tend-hconv-zconv, k=1')
#axs[0, 1].set_xlim([75,270])
#axs[0, 1].set_ylim([0,200])
g = axs[1, 0].pcolormesh(a[2,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-2e-7, -1e-7, 0, 1e-7,2e-7])
g = axs[1, 1].pcolormesh(a[3,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-2e-7, -1e-7, 0, 1e-7,2e-7])

In [ ]:
from matplotlib.colors import BoundaryNorm
a = res[0]
fig, axs = plt.subplots(2, 2, figsize=(10, 9))

vmax = 1
vmin = -1
cmap = plt.get_cmap('seismic')
levels = np.linspace(vmin, vmax, 22)  # num_levels bins
#levels = np.delete(levels,10)
norm = BoundaryNorm(boundaries=levels, ncolors=cmap.N, clip=True)

g = axs[0, 0].pcolormesh(a[0,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-1, 0, 1])
g = axs[0, 1].pcolormesh(a[1,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-1, 0, 1])
g = axs[1, 0].pcolormesh(a[2,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-1, 0, 1])
g = axs[1, 1].pcolormesh(a[3,0], cmap=cmap, norm=norm)
cbar = plt.colorbar(g)
cbar.set_ticks([-1, 0, 1])